# IRLR + SnAG Toy Experiment

Self-contained notebook to test the full IRLR pipeline on synthetic data:
1. **IRLR Module** — iterative mask refinement with per-iteration temperatures + entropy regularization
2. **SnAG PyramidBoundaryPredictor** — pyramid cls + reg head
3. **Combined loss** — mask deep supervision + dense (focal + GIoU) across pyramid levels

No external imports from the repo needed — everything is defined inline.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from dataclasses import dataclass

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Config

In [ ]:
@dataclass
class IRLRConfig:
    hidden_size: int = 256
    num_heads: int = 8
    num_z_tokens: int = 8
    R: int = 4
    ffn_ratio: int = 4
    dropout: float = 0.1
    sigma_max: float = 3.0
    alpha_mask: float = 1.0
    alpha_entropy: float = 0.1
    video_feat_dim: int = 2048
    text_word_dim: int = 300
    text_global_dim: int = 300

## 2. Mask Utilities

In [ ]:
def make_gt_mask(i0, i1, num_clips):
    device = i0.device
    clip_indices = torch.arange(num_clips, device=device).unsqueeze(0)
    start = i0.unsqueeze(1).float()
    end = i1.unsqueeze(1).float()
    gt_mask = ((clip_indices >= start) & (clip_indices <= end)).float()
    return gt_mask


def blur_mask(m_gt, sigma):
    if sigma <= 0:
        return m_gt
    kernel_size = int(6 * sigma + 1)
    if kernel_size % 2 == 0:
        kernel_size += 1
    x = torch.arange(kernel_size, dtype=m_gt.dtype, device=m_gt.device)
    x = x - kernel_size // 2
    kernel = torch.exp(-x ** 2 / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    kernel = kernel.view(1, 1, -1)
    padding = kernel_size // 2
    blurred = F.conv1d(m_gt.unsqueeze(1), kernel, padding=padding).squeeze(1)
    return blurred.clamp(0.0, 1.0)


def mask_entropy(m):
    mc = m.clamp(1e-6, 1 - 1e-6)
    return -(mc * mc.log() + (1 - mc) * (1 - mc).log()).mean()


def compute_mask_loss(masks, gt_mask, R, sigma_max, alpha_entropy=0.0):
    total_bce = 0.0
    total_entropy_reg = 0.0
    per_iter_losses = []

    for r in range(R):
        sigma = sigma_max * (1.0 - (r + 1) / R)
        target = blur_mask(gt_mask, sigma)
        iter_loss = F.binary_cross_entropy(masks[r], target)
        per_iter_losses.append(iter_loss.item())
        total_bce += iter_loss

        if alpha_entropy > 0:
            actual_ent = mask_entropy(masks[r])
            target_ent = 0.6 * (1.0 - (r + 1) / R) + 0.05
            total_entropy_reg += (actual_ent - target_ent) ** 2

    total_loss = total_bce / R + alpha_entropy * total_entropy_reg / R
    return total_loss, per_iter_losses

## 3. IRLR Module (with per-iteration mask temperatures)

In [ ]:
class MaskedCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.scale = self.d_head ** -0.5
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, kv, mask):
        B, K, _ = query.shape
        N = kv.shape[1]
        H = self.num_heads
        Q = self.W_Q(query).view(B, K, H, self.d_head).transpose(1, 2)
        Key = self.W_K(kv).view(B, N, H, self.d_head).transpose(1, 2)
        V = self.W_V(kv).view(B, N, H, self.d_head).transpose(1, 2)
        attn_logits = torch.matmul(Q, Key.transpose(-1, -2)) * self.scale
        mask_clamped = mask.clamp(min=1e-6)
        mask_bias = torch.log(mask_clamped).unsqueeze(1).unsqueeze(2)
        attn_logits = attn_logits + mask_bias
        attn_weights = F.softmax(attn_logits, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = torch.matmul(attn_weights, V)
        out = out.transpose(1, 2).contiguous().view(B, K, -1)
        out = self.W_O(out)
        return out


class IRLRSharedBlock(nn.Module):
    def __init__(self, d_model, num_heads, R=4, ffn_ratio=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.R = R
        self.W_a = nn.Linear(d_model, d_model, bias=False)
        nn.init.xavier_uniform_(self.W_a.weight, gain=0.1)

        # Per-iteration mask temperatures: soft -> sharp
        base = 1.0 / math.sqrt(d_model)
        init_temps = torch.linspace(0.3 * base, 1.0 * base, R)
        self.mask_scales = nn.Parameter(init_temps)

        self.masked_cross_attn = MaskedCrossAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.query_cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_ratio), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * ffn_ratio, d_model), nn.Dropout(dropout),
        )
        self.norm4 = nn.LayerNorm(d_model)

    def forward(self, Z, F_video, Q_words, iteration_idx, query_mask=None):
        Z_proj = self.W_a(Z)
        scores = torch.matmul(Z_proj, F_video.transpose(-1, -2)).mean(dim=1) * self.mask_scales[iteration_idx]
        m = torch.sigmoid(scores)

        cross_out = self.masked_cross_attn(Z, F_video, m)
        Z = self.norm1(Z + cross_out)

        self_out, _ = self.self_attn(Z, Z, Z)
        Z = self.norm2(Z + self_out)

        key_padding_mask = None
        if query_mask is not None:
            key_padding_mask = ~query_mask

        query_out, _ = self.query_cross_attn(Z, Q_words, Q_words, key_padding_mask=key_padding_mask)
        Z = self.norm3(Z + query_out)

        Z = self.norm4(Z + self.ffn(Z))
        return Z, m


class IRLRModule(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg.hidden_size

        self.video_proj = nn.Sequential(
            nn.Linear(cfg.video_feat_dim, d), nn.LayerNorm(d), nn.ReLU(), nn.Dropout(cfg.dropout),
        )
        self.text_word_proj = nn.Sequential(
            nn.Linear(cfg.text_word_dim, d), nn.LayerNorm(d), nn.ReLU(), nn.Dropout(cfg.dropout),
        )
        self.text_global_proj = nn.Sequential(
            nn.Linear(cfg.text_global_dim, d), nn.LayerNorm(d),
        )

        self.latent_embeds = nn.Parameter(torch.randn(cfg.num_z_tokens, d) * 0.02)
        self.W_q_init = nn.Linear(d, d)
        self.iter_embeds = nn.Parameter(torch.randn(cfg.R, 1, d) * 0.02)
        self.shared_block = IRLRSharedBlock(d, cfg.num_heads, R=cfg.R, ffn_ratio=cfg.ffn_ratio, dropout=cfg.dropout)

        self.reverse_cross_attn = nn.MultiheadAttention(d, cfg.num_heads, dropout=cfg.dropout, batch_first=True)
        self.reverse_norm = nn.LayerNorm(d)
        self.feature_combine = nn.Sequential(nn.Linear(d * 3, d), nn.GELU(), nn.Linear(d, d))
        self.output_norm = nn.LayerNorm(d)

    def forward(self, video_emb, query_tokens, text_emb, query_mask=None):
        F_video = self.video_proj(video_emb)
        Q_words = self.text_word_proj(query_tokens)
        q_global = self.text_global_proj(text_emb)
        B = F_video.shape[0]

        Z = self.latent_embeds.unsqueeze(0).expand(B, -1, -1) + self.W_q_init(q_global).unsqueeze(1)

        masks = []
        for r in range(self.cfg.R):
            Z_input = Z + self.iter_embeds[r]
            Z, m_r = self.shared_block(Z_input, F_video, Q_words, iteration_idx=r, query_mask=query_mask)
            masks.append(m_r)

        f_prime, _ = self.reverse_cross_attn(F_video, Z, Z)
        f_prime = self.reverse_norm(F_video + f_prime)

        m_final = masks[-1].unsqueeze(-1)
        combined = torch.cat([F_video, f_prime, m_final * F_video], dim=-1)
        F_enhanced = self.feature_combine(combined)
        F_enhanced = self.output_norm(F_enhanced + F_video)

        return F_enhanced, masks

## 4. SnAG Pyramid Head (inline)

In [ ]:
class MaskedConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, groups=1, bias=True):
        super().__init__()
        self.stride = stride
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=padding, groups=groups, bias=bias)
        if bias:
            nn.init.zeros_(self.conv.bias)

    def forward(self, x, mask):
        if mask is None:
            mask = torch.ones_like(x[:, :1], dtype=torch.bool)
        mask_float = mask.to(x.dtype)
        x = self.conv(x * mask_float)
        if self.stride > 1:
            mask_float = F.interpolate(mask_float, size=x.size(-1), mode='nearest')
            mask = mask_float.bool()
        return x, mask


class PyramidGenerator(nn.Module):
    def __init__(self, embd_dim, num_levels=3):
        super().__init__()
        self.num_levels = num_levels
        self.layers = nn.ModuleList()
        for _ in range(num_levels - 1):
            self.layers.append(MaskedConv1D(embd_dim, embd_dim, kernel_size=3, stride=2, padding=1, bias=False))

    def forward(self, x, mask):
        fpn, fpn_masks = [x], [mask]
        current_x, current_mask = x, mask
        for layer in self.layers:
            current_x, current_mask = layer(current_x, current_mask)
            current_x = F.relu(current_x)
            fpn.append(current_x)
            fpn_masks.append(current_mask)
        return tuple(fpn), tuple(fpn_masks)


class ChannelLayerNorm(nn.Module):
    def __init__(self, n_channels, eps=1e-5):
        super().__init__()
        self.n_channels = n_channels
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(n_channels, 1))
        self.bias = nn.Parameter(torch.zeros(n_channels, 1))

    def forward(self, x):
        x = x - torch.mean(x, dim=1, keepdim=True)
        sigma = torch.mean(x ** 2, dim=1, keepdim=True)
        x = x / torch.sqrt(sigma + self.eps)
        return x * self.weight + self.bias


class Scale(nn.Module):
    def __init__(self, init=1.0):
        super().__init__()
        self.scale = nn.Parameter(torch.as_tensor(init, dtype=torch.float))
    def forward(self, x):
        return x * self.scale.to(x.dtype)


class ClsHead(nn.Module):
    def __init__(self, embd_dim, n_layers=2, prior_prob=0.01):
        super().__init__()
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            self.convs.append(MaskedConv1D(embd_dim, embd_dim, 3, 1, 1, bias=False))
            self.norms.append(ChannelLayerNorm(embd_dim))
        self.cls_head = MaskedConv1D(embd_dim, 1, 3, 1, 1)
        bias_init = -np.log((1 - prior_prob) / prior_prob) if prior_prob > 0 else 0
        nn.init.constant_(self.cls_head.conv.bias, bias_init)

    def forward(self, fpn, fpn_masks):
        out_logits, out_masks = tuple(), tuple()
        for x, mask in zip(fpn, fpn_masks):
            for conv, norm in zip(self.convs, self.norms):
                x, _ = conv(x, mask)
                x = F.relu(norm(x), inplace=True)
            logits, _ = self.cls_head(x, mask)
            out_logits += (logits.squeeze(1),)
            out_masks += (mask.squeeze(1),)
        return out_logits, out_masks


class RegHead(nn.Module):
    def __init__(self, embd_dim, num_fpn_levels=1, n_layers=2):
        super().__init__()
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            self.convs.append(MaskedConv1D(embd_dim, embd_dim, 3, 1, 1, bias=False))
            self.norms.append(ChannelLayerNorm(embd_dim))
        self.reg_head = MaskedConv1D(embd_dim, 2, 3, 1, 1)
        self.scales = nn.ModuleList([Scale() for _ in range(num_fpn_levels)])

    def forward(self, fpn, fpn_masks):
        out_offsets, out_masks = tuple(), tuple()
        for i, (x, mask) in enumerate(zip(fpn, fpn_masks)):
            for conv, norm in zip(self.convs, self.norms):
                x, _ = conv(x, mask)
                x = F.relu(norm(x), inplace=True)
            offsets, _ = self.reg_head(x, mask)
            offsets = F.relu(self.scales[i](offsets))
            out_offsets += (offsets.transpose(1, 2),)
            out_masks += (mask.squeeze(1),)
        return out_offsets, out_masks


class PyramidBoundaryPredictor(nn.Module):
    def __init__(self, embd_dim=256, num_levels=3, n_layers=2):
        super().__init__()
        self.pyramid_gen = PyramidGenerator(embd_dim, num_levels)
        self.cls_head = ClsHead(embd_dim, n_layers=n_layers, prior_prob=0.01)
        self.reg_head = RegHead(embd_dim, num_fpn_levels=num_levels, n_layers=n_layers)

    def forward(self, video_feats, video_mask):
        fpn, fpn_masks = self.pyramid_gen(video_feats, video_mask)
        cls_logits, _ = self.cls_head(fpn, fpn_masks)
        reg_offsets, _ = self.reg_head(fpn, fpn_masks)
        return cls_logits, reg_offsets

## 5. End-to-End Model

In [ ]:
class IRLRWithSnAG(nn.Module):
    def __init__(self, cfg, num_pyramid_levels=3, head_n_layers=2):
        super().__init__()
        self.irlr = IRLRModule(cfg)
        self.snag_head = PyramidBoundaryPredictor(
            embd_dim=cfg.hidden_size,
            num_levels=num_pyramid_levels,
            n_layers=head_n_layers,
        )
        self.cfg = cfg
        self.num_pyramid_levels = num_pyramid_levels

    def forward(self, video_emb, query_tokens, text_emb, query_mask, video_mask):
        F_enhanced, masks = self.irlr(video_emb, query_tokens, text_emb, query_mask)
        F_transposed = F_enhanced.transpose(1, 2)             # (B, C, T)
        video_mask_3d = video_mask.unsqueeze(1).float()       # (B, 1, T)
        cls_logits, reg_offsets = self.snag_head(F_transposed, video_mask_3d)
        return cls_logits, reg_offsets, masks

## 6. Dense Loss Functions (Focal + GIoU)

In [ ]:
def sigmoid_focal_loss(inputs, targets, alpha=0.25, gamma=2.0, reduction='none'):
    inputs = inputs.float()
    targets = targets.float()
    mask = (targets >= 0.5).float()
    p = torch.sigmoid(inputs)
    p_t = p * targets + (1 - p) * (1 - targets)
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
    loss = ce_loss * (1 - p_t) ** gamma
    if alpha >= 0:
        alpha_t = alpha * mask + (1 - alpha) * (1 - mask)
        loss = alpha_t * loss
    if reduction == 'mean':
        loss = loss.mean()
    elif reduction == 'sum':
        loss = loss.sum()
    return loss


def ctr_giou_loss(input_offsets, target_offsets, reduction='none', eps=1e-8):
    input_offsets = input_offsets.float()
    target_offsets = target_offsets.float()
    lp, rp = input_offsets[:, 0], input_offsets[:, 1]
    lg, rg = target_offsets[:, 0], target_offsets[:, 1]
    lkis = torch.min(lp, lg)
    rkis = torch.min(rp, rg)
    intsctk = rkis + lkis
    unionk = (lp + rp) + (lg + rg) - intsctk
    iouk = intsctk / unionk.clamp(min=eps)
    loss = 1.0 - iouk
    if reduction == 'mean':
        loss = loss.mean() if loss.numel() > 0 else 0.0 * loss.sum()
    elif reduction == 'sum':
        loss = loss.sum()
    return loss


def get_center_sampling_mask(grid, gt_s_idx, gt_e_idx, radius=1.5):
    centers = ((gt_s_idx + gt_e_idx) / 2.0).unsqueeze(1)
    left_bound = centers - radius
    right_bound = centers + radius
    mask = (grid >= left_bound) & (grid <= right_bound)
    distances = torch.abs(grid - centers)
    closest_indices = distances.argmin(dim=1)
    closest_mask = torch.zeros_like(mask, dtype=torch.bool)
    closest_mask.scatter_(1, closest_indices.unsqueeze(1), True)
    mask = mask | closest_mask
    gt_mask = (grid >= gt_s_idx.unsqueeze(1)) & (grid <= gt_e_idx.unsqueeze(1))
    mask = mask & gt_mask
    return mask


def compute_dense_tr_loss(cls_logits, reg_offsets, gt_start, gt_end, video_mask,
                          sec_per_step=1.0, lambda_cls=1.0, lambda_reg=1.0, center_radius=1.5):
    B, T = cls_logits.shape
    device = cls_logits.device

    gt_s_idx = (gt_start / sec_per_step).round().float()
    gt_e_idx = (gt_end / sec_per_step).round().float()
    grid = torch.arange(T, device=device).unsqueeze(0).expand(B, T).float()

    pos_mask = get_center_sampling_mask(grid, gt_s_idx, gt_e_idx, radius=center_radius)
    pos_mask = pos_mask & video_mask.bool()
    target_cls = pos_mask.float()

    target_off_l = grid - gt_s_idx.unsqueeze(1)
    target_off_r = gt_e_idx.unsqueeze(1) - grid
    target_reg = torch.stack([target_off_l, target_off_r], dim=-1).float()

    valid_mask = video_mask.view(-1)
    cls_flat = cls_logits.view(-1)[valid_mask]
    target_cls_flat = target_cls.view(-1)[valid_mask]
    L_cls = sigmoid_focal_loss(cls_flat, target_cls_flat, alpha=0.25, gamma=2.0, reduction='sum')
    num_pos = pos_mask.sum().clamp(min=1.0)
    L_cls = L_cls / num_pos

    if pos_mask.sum() > 0:
        pred_reg_pos = reg_offsets[pos_mask]
        target_reg_pos = target_reg[pos_mask]
        L_reg = ctr_giou_loss(pred_reg_pos, target_reg_pos, reduction='sum')
        L_reg = L_reg / num_pos
    else:
        L_reg = torch.tensor(0.0, device=device)

    total_loss = lambda_cls * L_cls + lambda_reg * L_reg
    return total_loss, {'L_cls': L_cls, 'L_reg': L_reg, 'num_pos': num_pos}

## 7. Create Synthetic Batch

In [ ]:
def create_synthetic_batch(B=8, N=256, vid_dim=2048, word_dim=300, max_text_len=16):
    """Create a synthetic batch mimicking Charades-STA with I3D + GloVe."""
    video_emb = torch.randn(B, N, vid_dim)
    text_lens = torch.randint(5, max_text_len + 1, (B,))
    query_tokens = torch.zeros(B, max_text_len, word_dim)
    query_mask = torch.zeros(B, max_text_len, dtype=torch.bool)
    for i in range(B):
        l = text_lens[i].item()
        query_tokens[i, :l] = torch.randn(l, word_dim)
        query_mask[i, :l] = True
    text_emb = query_tokens.sum(dim=1) / text_lens.unsqueeze(1).float()

    video_mask = torch.ones(B, N, dtype=torch.bool)

    # Random ground-truth segments
    i0 = torch.randint(10, N // 2, (B,))
    i1 = i0 + torch.randint(10, N // 3, (B,))
    i1 = i1.clamp(max=N - 1)

    duration = torch.full((B,), 30.0)  # 30 sec videos
    start_sec = (i0.float() / N) * duration
    end_sec = (i1.float() / N) * duration

    return {
        'video_emb': video_emb,
        'query_tokens': query_tokens,
        'text_emb': text_emb,
        'query_mask': query_mask,
        'video_mask': video_mask,
        'i0': i0,
        'i1': i1,
        'duration': duration,
        'start_sec': start_sec,
        'end_sec': end_sec,
    }


batch = create_synthetic_batch(B=8)
print('Batch created:')
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:20s}: {list(v.shape)}')

## 8. Build Model & Verify Forward Pass

In [ ]:
cfg = IRLRConfig(
    hidden_size=256,
    num_heads=8,
    num_z_tokens=8,
    R=4,
    sigma_max=3.0,
    alpha_mask=1.0,
    alpha_entropy=0.1,
    video_feat_dim=2048,
    text_word_dim=300,
    text_global_dim=300,
)

model = IRLRWithSnAG(cfg, num_pyramid_levels=3, head_n_layers=2).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
irlr_params = sum(p.numel() for p in model.irlr.parameters() if p.requires_grad)
snag_params = sum(p.numel() for p in model.snag_head.parameters() if p.requires_grad)
print(f'Total params:  {total_params:,}')
print(f'  IRLR params: {irlr_params:,}')
print(f'  SnAG params: {snag_params:,}')

# Verify forward
model.eval()
with torch.no_grad():
    cls_logits, reg_offsets, masks = model(
        batch['video_emb'].to(device),
        batch['query_tokens'].to(device),
        batch['text_emb'].to(device),
        batch['query_mask'].to(device),
        batch['video_mask'].to(device),
    )

print(f'\nPyramid levels: {len(cls_logits)}')
for i, (c, r) in enumerate(zip(cls_logits, reg_offsets)):
    print(f'  Level {i}: cls={list(c.shape)}, reg={list(r.shape)}')
print(f'Masks: {len(masks)} iterations, each {list(masks[0].shape)}')

# Check initial mask temperatures
print(f'\nMask scales (per iteration): {model.irlr.shared_block.mask_scales.data.tolist()}')
for r, m in enumerate(masks):
    ent = mask_entropy(m).item()
    print(f'  Iter {r}: mean={m.mean():.3f}, std={m.std():.3f}, entropy={ent:.3f}')

## 9. Training Loop (Mask + Dense Loss)

In [ ]:
def run_toy_experiment(batch, cfg, device='cuda', num_train_steps=100, lr=5e-4,
                       num_pyramid_levels=3):
    # Move to device
    video_emb = batch['video_emb'].to(device)
    query_tokens = batch['query_tokens'].to(device)
    text_emb = batch['text_emb'].to(device)
    query_mask = batch['query_mask'].to(device)
    video_mask = batch['video_mask'].to(device)
    i0 = batch['i0'].to(device)
    i1 = batch['i1'].to(device)

    B, N = video_emb.shape[:2]
    gt_mask = make_gt_mask(i0, i1, N).to(device)

    # Model
    model = IRLRWithSnAG(cfg, num_pyramid_levels=num_pyramid_levels).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # --- Before training ---
    model.eval()
    with torch.no_grad():
        cls_logits, reg_offsets, masks = model(video_emb, query_tokens, text_emb, query_mask, video_mask)
        mask_loss_pre, _ = compute_mask_loss(masks, gt_mask, cfg.R, cfg.sigma_max, cfg.alpha_entropy)
    print(f'Before training: mask_loss={mask_loss_pre.item():.4f}')
    for r, m in enumerate(masks):
        ent = mask_entropy(m).item()
        print(f'  Iter {r}: mean={m.mean():.3f}, std={m.std():.3f}, entropy={ent:.3f}')

    # --- Train ---
    print(f'\n--- Training for {num_train_steps} steps ---')
    model.train()
    history = {'total': [], 'mask': [], 'dense': [], 'L_cls': [], 'L_reg': []}
    mask_snapshots = []  # (step, [mask_per_iter])
    entropy_history = {r: [] for r in range(cfg.R)}  # track entropy per iter

    for step in range(num_train_steps):
        cls_logits, reg_offsets, masks = model(video_emb, query_tokens, text_emb, query_mask, video_mask)

        # Mask loss
        mask_loss, per_iter = compute_mask_loss(masks, gt_mask, cfg.R, cfg.sigma_max, cfg.alpha_entropy)

        # Dense loss across pyramid
        total_dense = torch.tensor(0.0, device=device)
        dense_info_agg = {'L_cls': 0.0, 'L_reg': 0.0}
        for lvl, (lc, lo) in enumerate(zip(cls_logits, reg_offsets)):
            stride = 2 ** lvl
            if lvl == 0:
                lvl_mask = video_mask
            else:
                lvl_mask = F.interpolate(
                    video_mask.unsqueeze(1).float(), size=lc.shape[1], mode='nearest'
                ).squeeze(1).bool()
            dl, di = compute_dense_tr_loss(lc, lo, i0.float(), i1.float(), lvl_mask, sec_per_step=stride)
            total_dense = total_dense + dl
            dense_info_agg['L_cls'] += di['L_cls'].item()
            dense_info_agg['L_reg'] += di['L_reg'].item()

        total_loss = cfg.alpha_mask * mask_loss + total_dense

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        history['total'].append(total_loss.item())
        history['mask'].append(mask_loss.item())
        history['dense'].append(total_dense.item())
        history['L_cls'].append(dense_info_agg['L_cls'])
        history['L_reg'].append(dense_info_agg['L_reg'])

        # Track entropy per iteration
        with torch.no_grad():
            for r in range(cfg.R):
                entropy_history[r].append(mask_entropy(masks[r]).item())

        if step % 10 == 0 or step == num_train_steps - 1:
            with torch.no_grad():
                snap_masks = [m.clone() for m in masks]
                mask_snapshots.append((step, snap_masks))
            ent_str = ', '.join([f'{entropy_history[r][-1]:.3f}' for r in range(cfg.R)])
            print(f'  Step {step:3d}: total={total_loss.item():.4f} '
                  f'mask={mask_loss.item():.4f} dense={total_dense.item():.4f} '
                  f'| entropy=[{ent_str}]')

    # --- After training ---
    print(f'\n--- After Training ---')
    model.eval()
    with torch.no_grad():
        cls_logits, reg_offsets, masks = model(video_emb, query_tokens, text_emb, query_mask, video_mask)
        mask_loss_post, per_iter_post = compute_mask_loss(masks, gt_mask, cfg.R, cfg.sigma_max, cfg.alpha_entropy)

    print(f'Mask loss: {mask_loss_pre.item():.4f} -> {mask_loss_post.item():.4f}')
    print(f'Mask scales (learned): {model.irlr.shared_block.mask_scales.data.tolist()}')
    for r, m in enumerate(masks):
        sigma = cfg.sigma_max * (1.0 - (r + 1) / cfg.R)
        ent = mask_entropy(m).item()
        print(f'  Iter {r} (target sigma={sigma:.2f}): mean={m.mean():.3f}, std={m.std():.3f}, entropy={ent:.3f}')

    return model, masks, gt_mask, history, mask_snapshots, entropy_history

In [ ]:
model, masks, gt_mask, history, mask_snapshots, entropy_history = run_toy_experiment(
    batch, cfg, device=device, num_train_steps=100, lr=5e-4
)

## 10. Visualization

In [ ]:
# --- Loss curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['total'], label='Total', linewidth=2)
axes[0].plot(history['mask'], label='Mask', linewidth=1.5, alpha=0.8)
axes[0].plot(history['dense'], label='Dense', linewidth=1.5, alpha=0.8)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['L_cls'], label='Focal (cls)', linewidth=1.5)
axes[1].plot(history['L_reg'], label='GIoU (reg)', linewidth=1.5)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_title('Dense Loss Breakdown'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, cfg.R))
for r in range(cfg.R):
    sigma = cfg.sigma_max * (1.0 - (r + 1) / cfg.R)
    target_ent = 0.6 * (1.0 - (r + 1) / cfg.R) + 0.05
    axes[2].plot(entropy_history[r], label=f'Iter {r} (target={target_ent:.2f})',
                 linewidth=1.5, color=colors[r])
    axes[2].axhline(y=target_ent, color=colors[r], linestyle='--', alpha=0.4)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Entropy')
axes[2].set_title('Mask Entropy per Iteration'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('toy_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: toy_loss_curves.png')

In [ ]:
# --- Mask evolution across iterations (final trained model) ---
sample_idx = 0
N = gt_mask.shape[1]
x = np.arange(N)

fig, axes = plt.subplots(cfg.R + 1, 1, figsize=(14, 3 * (cfg.R + 1)), sharex=True)

# GT
axes[0].fill_between(x, 0, gt_mask[sample_idx].cpu().numpy(),
                     alpha=0.3, color='green', label='Ground Truth')
axes[0].set_ylabel('Relevance'); axes[0].set_title('Ground Truth Mask')
axes[0].legend(); axes[0].set_ylim(-0.05, 1.15)

for r in range(cfg.R):
    ax = axes[r + 1]
    sigma = cfg.sigma_max * (1.0 - (r + 1) / cfg.R)
    blurred_target = blur_mask(gt_mask, sigma)[sample_idx].cpu().numpy()
    predicted = masks[r][sample_idx].detach().cpu().numpy()

    ax.fill_between(x, 0, blurred_target, alpha=0.2, color='blue',
                    label=f'Target (sigma={sigma:.2f})')
    ax.plot(x, predicted, color='red', linewidth=1.5, label='Predicted')
    ax.fill_between(x, 0, gt_mask[sample_idx].cpu().numpy(),
                    alpha=0.1, color='green', label='GT (ref)')
    ent = mask_entropy(masks[r]).item()
    scale = model.irlr.shared_block.mask_scales[r].item()
    ax.set_ylabel('Relevance')
    ax.set_title(f'Iter {r} | sigma={sigma:.2f} | entropy={ent:.3f} | scale={scale:.5f}')
    ax.legend(loc='upper right'); ax.set_ylim(-0.05, 1.15)

axes[-1].set_xlabel('Clip Index')
plt.tight_layout()
plt.savefig('toy_masks_per_iteration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: toy_masks_per_iteration.png')

In [ ]:
# --- Mask training progress (final iteration mask at different training steps) ---
num_snaps = len(mask_snapshots)
fig, axes = plt.subplots(num_snaps, 1, figsize=(14, 2.5 * num_snaps), sharex=True)
if num_snaps == 1:
    axes = [axes]

for idx, (step_num, snap_masks) in enumerate(mask_snapshots):
    ax = axes[idx]
    # Show all R iterations at this training step
    ax.fill_between(x, 0, gt_mask[sample_idx].cpu().numpy(),
                    alpha=0.15, color='green', label='GT')
    for r in range(cfg.R):
        predicted = snap_masks[r][sample_idx].cpu().numpy()
        ax.plot(x, predicted, linewidth=1.2, alpha=0.8, label=f'Iter {r}')
    ax.set_ylabel('Score')
    ax.set_title(f'Step {step_num}: all iterations')
    ax.legend(loc='upper right', fontsize=8); ax.set_ylim(-0.05, 1.15)

axes[-1].set_xlabel('Clip Index')
plt.tight_layout()
plt.savefig('toy_training_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: toy_training_progress.png')

## 11. Coarse-to-Fine Analysis

In [ ]:
print('--- Coarse-to-Fine Analysis ---')
print(f'{"Iter":>4} {"sigma":>6} {"entropy":>8} {"P":>6} {"R":>6} {"F1":>6} {"scale":>10}')
print('-' * 55)

for r, m in enumerate(masks):
    sigma = cfg.sigma_max * (1.0 - (r + 1) / cfg.R)
    ent = mask_entropy(m).item()
    mb = (m > 0.5).float()
    p = (mb * gt_mask).sum() / (mb.sum() + 1e-6)
    rc = (mb * gt_mask).sum() / (gt_mask.sum() + 1e-6)
    f1 = 2 * p * rc / (p + rc + 1e-6)
    scale = model.irlr.shared_block.mask_scales[r].item()
    print(f'{r:4d} {sigma:6.2f} {ent:8.3f} {p:6.3f} {rc:6.3f} {f1:6.3f} {scale:10.6f}')

print('\nExpected: entropy DECREASES, F1 INCREASES across iterations')

## 12. Compare: WITH vs WITHOUT entropy regularization

In [ ]:
# Run WITHOUT entropy reg for comparison
cfg_no_ent = IRLRConfig(
    hidden_size=256, num_heads=8, num_z_tokens=8, R=4,
    sigma_max=3.0, alpha_mask=1.0, alpha_entropy=0.0,  # <-- disabled
    video_feat_dim=2048, text_word_dim=300, text_global_dim=300,
)

print('=== WITHOUT entropy regularization (alpha_entropy=0) ===')
_, masks_no_ent, _, _, _, ent_hist_no = run_toy_experiment(
    batch, cfg_no_ent, device=device, num_train_steps=100, lr=5e-4
)

print('\n=== WITH entropy regularization (alpha_entropy=0.1) ===')
# Already ran above, just print summary
for r in range(cfg.R):
    ent_with = mask_entropy(masks[r]).item()
    ent_without = mask_entropy(masks_no_ent[r]).item()
    print(f'  Iter {r}: entropy WITH={ent_with:.3f}, WITHOUT={ent_without:.3f}')

In [ ]:
# Side-by-side entropy comparison plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, cfg.R))

for r in range(cfg.R):
    target_ent = 0.6 * (1.0 - (r + 1) / cfg.R) + 0.05
    axes[0].plot(ent_hist_no[r], label=f'Iter {r}', color=colors[r], linewidth=1.5)
    axes[1].plot(entropy_history[r], label=f'Iter {r}', color=colors[r], linewidth=1.5)
    axes[1].axhline(y=target_ent, color=colors[r], linestyle='--', alpha=0.4)

axes[0].set_title('WITHOUT entropy reg (alpha=0)')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Entropy')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(0, 0.75)

axes[1].set_title('WITH entropy reg (alpha=0.1)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Entropy')
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 0.75)

plt.tight_layout()
plt.savefig('toy_entropy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: toy_entropy_comparison.png')